In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
# # %%captureo
# import torch
# major_version, minor_version = torch.cuda.get_device_capability()
# if major_version >= 8:
#     !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# else:
#     !pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [ ]:
# pip install "unsloth[cu124-torch240] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
# Prevention for memory fragmentation

Model

In [ ]:
max_seq_length = 2048 # Adjust based on legal document lengths
dtype = None # Auto-detection
load_in_4bit = True # Essential for T4 GPUs

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters for DAPT
model = FastLanguageModel.get_peft_model(
    model,
     # Higher rank -> dapt helps absorb more domain knowledge
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Load Pre-tokenized Dataset

In [ ]:
from datasets import load_dataset, load_from_disk

# Load your uploaded Parquet file
# dataset = load_dataset("parquet", data_files={"train": "../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_1990-2025.parquet"}, split="train")

# New tokenized dataset
dataset = load_from_disk("pre_tokenized_dataset")

In [ ]:
print(dataset[0].keys())
# Should show dict_keys(['input_ids', 'attention_mask'])

Domain Adaptive Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "input_ids", 
    max_seq_length = max_seq_length,
    packing = False, # Set to False; already packed the data
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, 
        warmup_steps = 20,
        # max_steps = (len(dataset) // (1 * 8)),
        max_steps = 200, 
        learning_rate = 2e-4, 
        
        # fp16 = not torch.cuda.is_bf16_supported(),
        # bf16 = torch.cuda.is_bf16_supported(),

        fp16 = False,
        bf16 = True,

        
        logging_steps = 1,
        optim = "paged_adamw_8bit", # Standard for dual T4
        weight_decay = 0.01,
        save_strategy = "steps",
        report_to = "none",

    ),
)

In [ ]:
trainer.train()

In [ ]:
# # This saves ONLY the trained adapter layers (very small and disk-safe)
model.save_pretrained("Deepseek_R1_8B_DAPT_LoRA")
tokenizer.save_pretrained("Deepseek_R1_8B_DAPT1_LoRA")

In [ ]:
# Merge to 16bit and save as GGUF for Ollama
model.save_pretrained_gguf("rtg_deepseek_dapt_model", tokenizer, quantization_method = "q4_k_m")
